# Block 01: Claim Mapping and Execution Rules

**Goal**  
Freeze the experiment-to-claim map against the current paper checklist before numerical work.

**Checklist alignment**  
Maps E1-E12 to regimes, metrics, and artifact destinations.

**Outputs**  
- tables under `results/tables/`
- figures under `results/figures/`
- manifests under `results/manifests/`
- executed notebook copy under `results/notebooks/` when this suite is run with `--execute`

In [1]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "experiment_checklist.md").exists() and (candidate / "implementation").exists():
            return candidate
    raise RuntimeError("Could not locate repo root for notebook execution.")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from implementation.notebook_support import *

cfg = CanonicalConfig().validate()
dirs = ensure_results_dirs(cfg)
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

In [2]:
claim_map = dataframe_from_claim_map()
naming_rules = pd.DataFrame(
    [
        ("preprocessing", "baseline_mean__rfft__one_sided_psd"),
        ("simulator_rank1", "QPSimulator aligned arrivals + external stationary noise"),
        ("simulator_family", "QPSimulator controlled tau/t0/n_QP family"),
        ("rank_label", "k{integer}"),
        ("seed_label", "seed_{integer}"),
        ("result_notebook", "results/notebooks/block_XX_*.ipynb"),
    ],
    columns=["name", "value"],
)
display(claim_map)
display(naming_rules)

,experiment_id,claim,paper_anchor,support_regime,metrics,artifact_path
0,E1,rank-1 EMPCA ≡ OF amplitude estimator,04_empca / theorem 1,theorem-support,weighted cosine; amplitude correlation; median relative amplitude error; residual KS p-value,results/tables/block_04_e1_rank1_summary.csv
1,E2,noise-aware linear AE ≡ EMPCA,05_linear_ae / theorem 2,theorem-support,principal-angle cosines for k=1..3; residual mean gap; residual KS p-value,results/tables/block_05_e2_bridge.csv
2,E3,chi^2(k) decreases monotonically with rank,04_empca / rank-k proposition,theorem-support,held-out weighted residual by k for 1D and multi-D families,results/tables/block_06_e3_monotonicity.csv
3,E4,CRB variance law,03_optimal_filter / CRB,theorem-support,empirical amplitude variance vs 1 / N_Phi across noise colors,results/tables/block_04_e4_crb.csv
4,E5,energy-resolution scaling law,03_optimal_filter / resolution,mixed,sigma_E vs noise power slope; simulated residuals; real-data sigma comparison,results/tables/block_04_e5_resolution.csv
5,E6,noise-aware EMPCA beats isotropic PCA under colored noise,07_experiments / ablation,theorem-support,"weighted residual gain under white, pink, brownian noise",results/tables/block_07_e6_ablation.csv
6,E7,template mismatch bias and rank-2 recovery,03_optimal_filter / template mismatch,theorem-support,mean amplitude bias; RMSE; bias-vs-cos^2 curve,results/tables/block_07_e7_mismatch.csv
7,E8,time-shift OF recovers jittered arrivals,03_optimal_filter / shifted OF,theorem-support,arrival-time RMSE; amplitude RMSE; fixed-vs-shift bias,results/tables/block_07_e8_time_shift.csv
8,E9,EMPCA convergence diagnostics,06_convergence / convergence theorem,theorem-support,chi^2 trace monotonicity; iterations to tolerance; init dependence,results/tables/block_06_e9_convergence.csv
9,E10,non-stationary noise robustness,06_convergence / stationarity discussion,robustness-support,global-vs-segment PSD RMSE and weighted residual,results/tables/block_09_e10_nonstationary.csv


,name,value
0,preprocessing,baseline_mean__rfft__one_sided_psd
1,simulator_rank1,QPSimulator aligned arrivals + external stationary noise
2,simulator_family,QPSimulator controlled tau/t0/n_QP family
3,rank_label,k{integer}
4,seed_label,seed_{integer}
5,result_notebook,results/notebooks/block_XX_*.ipynb


In [3]:
claim_map_path = dirs["tables"] / "block_01_claim_map.csv"
rules_path = dirs["manifests"] / "block_01_execution_rules.json"
save_dataframe(claim_map, claim_map_path)
save_json(
    {
        "canonical_config": config_as_frame(cfg).to_dict(orient="records"),
        "naming_rules": naming_rules.to_dict(orient="records"),
    },
    rules_path,
)
pd.DataFrame(
    [
        manifest_row("block_01", "governance", str(claim_map_path.relative_to(REPO_ROOT)), cfg),
        manifest_row("block_01", "governance", str(rules_path.relative_to(REPO_ROOT)), cfg),
    ]
)

,block_id,regime,output_path,seed,trace_len,pretrigger
0,block_01,governance,results/tables/block_01_claim_map.csv,314159,32768,4000
1,block_01,governance,results/manifests/block_01_execution_rules.json,314159,32768,4000
